In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, LeakyReLU, Activation
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import legacy, Adam, RMSprop
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.models import Model
from sklearn.metrics import classification_report

In [ ]:
# Load in images

# Set the path to your dataset directory
dataset_path = 'Data/Training_Data'

# Create an ImageDataGenerator object
train_datagen = ImageDataGenerator(dtype = 'float32', preprocessing_function=preprocess_input, validation_split=0.2)

# Set the batch size
batch_size = 32

# Load training data from directory
train_data = train_datagen.flow_from_directory(
    directory=dataset_path,
    target_size=(224, 224),  # Resize all images to 224x224
    batch_size=batch_size,
    class_mode='binary',  # Use 'binary' for binary classification problems
    shuffle=True,
    seed=42,  # For reproducibility
    subset='training',  # For training
)

# Load validation data from the same directory (20% of the data)
val_data = train_datagen.flow_from_directory(
    directory=dataset_path,
    target_size=(224, 224),
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False, # optional for validation
    # seed=42,
    subset='validation',  # For validation
)

In [ ]:
# check if any filenames same in both training and validation data
train_files = set(train_data.filenames)
val_files = set(val_data.filenames)
common_files = train_files.intersection(val_files)
print(f'Number of common files in training and validation data: {len(common_files)}')

In [ ]:
# check images data type
print(train_data[0][0].dtype)

In [ ]:
# print class names for each dataset
print('Training Data Classes:', train_data.class_indices)
print('Validation Data Classes:', val_data.class_indices)

In [ ]:
# Print class distribution for each dataset

# Training Data
train_class_distribution = pd.Series(train_data.classes).value_counts()
print('Training Data Class Distribution:')
print(train_class_distribution)

# Validation Data
val_class_distribution = pd.Series(val_data.classes).value_counts()
print('Validation Data Class Distribution:')
print(val_class_distribution)

## Model Building and Training

In [ ]:
# Load the ResNet50 model (or replace with other models to test)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model
base_model.trainable = False

# Hyperparameter Tuning: Try different values to determine best model

# Build Model
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(1024, activation='relu', kernel_regularizer=l2(0.01))(x)  # L2 regularization
x = Dense(1024, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer=legacy.Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Model summary 
model.summary()

In [ ]:
# train model with callbacks

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.00001, verbose=1)

history = model.fit(
    train_data,
    epochs=100,
    validation_data=val_data,
    callbacks=[early_stopping, reduce_lr]
)

In [ ]:
# evaluate the model
test_loss, test_accuracy = model.evaluate(val_data)
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
# plot training and validation accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# plot training and validation loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# get model evaluation metrics: F1 score, precision, recall

# make predictions on the test set
predictions = model.predict(val_data)
# convert predictions to binary
binary_predictions = np.where(predictions > 0.5, 1, 0)

# get classification report
report = classification_report(val_data.classes, binary_predictions)
print(report)

In [ ]:
# import seaborn and confusion matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix

# get confusion matrix
cm = confusion_matrix(val_data.classes, binary_predictions)

# plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=val_data.class_indices, yticklabels=val_data.class_indices)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# get true positive, true negative, false positive, false negative
TP = cm[1][1]
TN = cm[0][0]
FP = cm[0][1]
FN = cm[1][0]

# calculate sensitivity and specificity
sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)

print(f'Sensitivity: {sensitivity}')
print(f'Specificity: {specificity}')

In [ ]:
# get AUC-ROC score
from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(val_data.classes, binary_predictions)
print(f'AUC-ROC Score: {roc_auc}')

In [ ]:
# save model as resunet_model in tf format

model.save('resnet_model_final', save_format='tf')

## Additional Resutls

In [ ]:
# plot ROC curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(val_data.classes, binary_predictions)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.show()

In [ ]:
# create function to calculate Matthew’s correlation coefficient

def calculate_mcc(y_true, y_pred):
    conf_matrix = confusion_matrix(y_true, y_pred)
    tp = conf_matrix[0, 0]
    tn = conf_matrix[1, 1]
    fp = conf_matrix[0, 1]
    fn = conf_matrix[1, 0]
    mcc = (tp * tn - fp * fn) / np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return mcc

In [ ]:
# use function to calculate MCC
mcc = calculate_mcc(val_data.classes, binary_predictions)
print(f'MCC: {mcc}')

In [ ]:
# create function to find mcc of each class (0 and 1) separately and also an average mcc


def calculate_mcc_per_class(y_true, y_pred):
    conf_matrix = confusion_matrix(y_true, y_pred)
    # For Class 0
    tp0 = conf_matrix[0, 0]  # True Positive for class 0
    tn0 = conf_matrix[1, 1]  # True Negative for class 0
    fp0 = conf_matrix[0, 1]  # False Positive for class 0
    fn0 = conf_matrix[1, 0]  # False Negative for class 0
    
    denominator0 = np.sqrt((tp0 + fp0) * (tp0 + fn0) * (tn0 + fp0) * (tn0 + fn0))
    mcc0 = (tp0 * tn0 - fp0 * fn0) / denominator0 if denominator0 != 0 else 0  # Check for division by zero

    # For Class 1
    tp1 = conf_matrix[1, 1]  # True Positive for class 1
    tn1 = conf_matrix[0, 0]  # True Negative for class 1
    fp1 = conf_matrix[1, 0]  # False Positive for class 1
    fn1 = conf_matrix[0, 1]  # False Negative for class 1
    
    denominator1 = np.sqrt((tp1 + fp1) * (tp1 + fn1) * (tn1 + fp1) * (tn1 + fn1))
    mcc1 = (tp1 * tn1 - fp1 * fn1) / denominator1 if denominator1 != 0 else 0  # Check for division by zero

    # Average MCC
    avg_mcc = (mcc0 + mcc1) / 2

    return mcc0, mcc1, avg_mcc

# Example usage:
mcc0, mcc1, avg_mcc = calculate_mcc_per_class(val_data.classes, binary_predictions)
print(f'MCC for class 0: {mcc0}')
print(f'MCC for class 1: {mcc1}')
print(f'Average MCC: {avg_mcc}')
